# 04  Backtest & Strategy Comparison

Aggregates the per-fold predictions saved by `scripts/03_train_dmn.py` (or `scripts/03bis_walk_forward.py`) into a continuous out-of-sample track record, computes the paper's full performance table, and benchmarks the DMN variants against simple references.

**Variants compared:**
- DMN (LSTM only), paper's baseline (`USE_CPD=False`)
- DMN w/ CPD, paper's main contribution (`USE_CPD=True`)
- DMN long-only w/ CPD, positions in $(0, 1)$
- MACD, Baz et al. 2015, single-asset signal averaged across stocks
- TSMOM (Moskowitz), sgn(252-day return), vol-scaled
- EW-SXXR, equal-weighted portfolio 
- SXXR, cap-weighted reference index

**Metrics (paper Exhibits 3/4)**

Annualised return, vol, Sharpe; downside deviation and Sortino; max drawdown and Calmar; hit rate; profit-to-loss ratio. 
Reported both for the raw signal output and after rescaling to the 15% volatility target (with and without the 25 bps transaction-cost adjustment)

In [ ]:
from pathlib import Path
import sys, warnings, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (14, 5), "figure.dpi": 100, "axes.grid": True})

## Configuration

In [ ]:
with open(PROJECT_ROOT / "configs" / "default.yaml") as f:
    cfg = yaml.safe_load(f)

PROCESSED_DIR = PROJECT_ROOT / cfg["data"]["processed_dir"]
MODELS_DIR    = PROCESSED_DIR / "models"
TARGET_VOL    = cfg["vol_target"]
CPD_LBW       = cfg["dmn"]["cpd_lbw"]

# Transaction cost (Appendix C of the paper). 0 = paper main result; 0.0025 = 25 bps for the realistic deployment scenario
TRANSACTION_COST = 0.0025

print(f"Models dir:      {MODELS_DIR}")
print(f"Target vol:      {TARGET_VOL:.0%}")
print(f"Transaction cost: {TRANSACTION_COST*1e4:.0f} bps")

## Load DMN predictions
 
Predictions from each fold are concatenated into a single continuous out-of-sample series. 
We support multiple variants in parallel — the variant suffix (e.g. `cpd21`, `nocpd`, `cpd21_longonly`) is the discriminator.

In [ ]:
def load_variant(variant: str) -> pd.DataFrame:
    """Concatenate predictions across all folds for a given DMN variant."""
    files = sorted(MODELS_DIR.glob(f"predictions_fold*_{variant}.csv"))
    if not files:
        return pd.DataFrame()
    frames = [pd.read_csv(f, parse_dates=["date"]) for f in files]
    df = pd.concat(frames, ignore_index=True).sort_values(["date", "ticker"])
    df["variant"] = variant
    return df

# Load whichever variants exist on disk; missing ones are skipped silently
VARIANTS = {
    "dmn_baseline":     "nocpd",
    "dmn_cpd":          f"cpd{CPD_LBW}",
    "dmn_cpd_longonly": f"cpd{CPD_LBW}_longonly",
}

dmn_predictions = {}
for label, suffix in VARIANTS.items():
    df = load_variant(suffix)
    if df.empty:
        print(f"  {label}: no predictions found ({suffix}.csv) — skipping")
    else:
        dmn_predictions[label] = df
        print(f"  {label}: {len(df):,} rows, "
              f"{df['date'].min().date()}{df['date'].max().date()}")

if not dmn_predictions:
    raise FileNotFoundError(
        "No DMN predictions found. Run scripts/03bis_walk_forward.py first."
    )

## Load benchmarks and the processed panel
 
We need the panel to compute the classical TSMOM and MACD reference signals on the same dates as the DMN test predictions, ensuring a fair comparison.

In [ ]:
benchmarks = pd.read_csv(PROCESSED_DIR / "benchmark_stoxx600_ew.csv",
                         parse_dates=["date"])
print(f"Benchmarks: {len(benchmarks):,} rows, "
      f"{benchmarks['benchmark'].unique().tolist()}")

stocks = pd.read_csv(
    PROCESSED_DIR / "stoxx600_processed.csv",
    parse_dates=["date"],
    usecols=["date", "ticker", "1d_arith_ret", "60d_ewm_vol",
             "252d_arith_ret", "macd_8_24", "macd_16_48", "macd_32_96"],
)

# Restrict the panel to the out-of-sample period covered by the DMN
oos_start = min(df["date"].min() for df in dmn_predictions.values())
oos_end   = max(df["date"].max() for df in dmn_predictions.values())
stocks_oos = stocks.loc[(stocks["date"] >= oos_start) &
                         (stocks["date"] <= oos_end)].copy()
print(f"Out-of-sample period: {oos_start.date()}{oos_end.date()}  "
      f"({stocks_oos['date'].nunique():,} days, "
      f"{stocks_oos['ticker'].nunique():,} tickers)")

## Build paper's reference strategies

In [ ]:
def vol_scaled_strategy_return(positions, ret_real, ex_ante_vol, target_vol=TARGET_VOL):
    """Apply paper Eq. 11 vol scaling to per-(stock, date) positions."""
    return positions * (target_vol / np.maximum(ex_ante_vol, 1e-6)) * ret_real

# Long-only (= EW)
long_only_pos = pd.Series(1.0, index=stocks_oos.index)

# Moskowitz: sign of trailing 252-day return
moskowitz_pos = np.sign(stocks_oos["252d_arith_ret"].fillna(0.0))

# MACD: combined signal (mean across the three (S, L) pairs), then sign for binary trade
macd_signal = stocks_oos[["macd_8_24", "macd_16_48", "macd_32_96"]].mean(axis=1)
macd_pos    = np.sign(macd_signal.fillna(0.0))

stocks_oos["pos_long_only"] = long_only_pos
stocks_oos["pos_moskowitz"] = moskowitz_pos
stocks_oos["pos_macd"]      = macd_pos

# Vol-scaled returns for each strategy
for name in ["long_only", "moskowitz", "macd"]:
    stocks_oos[f"strat_{name}"] = vol_scaled_strategy_return(
        stocks_oos[f"pos_{name}"],
        stocks_oos["1d_arith_ret"].shift(-1),  # next-day return (held over the position)
        stocks_oos["60d_ewm_vol"],
    )

# Drop rows where strategy returns can't be computed
classical = (stocks_oos.dropna(subset=["strat_long_only", "strat_moskowitz", "strat_macd"])
                       [["date", "ticker", "pos_long_only", "pos_moskowitz", "pos_macd",
                         "strat_long_only", "strat_moskowitz", "strat_macd"]])
print(f"Classical strategies computed on {len(classical):,} (stock, date) pairs")

## Apply transaction costs
 
Following paper Eq. C1, the cost is proportional to the change in the vol-scaled position $X_t / \sigma_t$ between consecutive periods. We compute it per (ticker, date) and aggregate later.

In [ ]:
def add_transaction_costs(df: pd.DataFrame, position_col: str, vol_col: str,
                          strat_col: str, cost: float = TRANSACTION_COST,
                          target_vol: float = TARGET_VOL) -> pd.DataFrame:
    """Subtract cost * target_vol * |Δ(X_t / σ_t)| from the strategy return.
    
    Operates per ticker (the cost is on each stock's individual turnover).
    Returns a copy of df with a new column f"{strat_col}_net".
    """
    df = df.sort_values(["ticker", "date"]).copy()
    df["scaled_pos"] = df[position_col] / np.maximum(df[vol_col], 1e-6)
    df["d_scaled_pos"] = df.groupby("ticker")["scaled_pos"].diff().fillna(0.0)
    df[f"{strat_col}_net"] = df[strat_col] - cost * target_vol * df["d_scaled_pos"].abs()
    return df.drop(columns=["scaled_pos", "d_scaled_pos"])

# Apply to DMN variants
for label, df in dmn_predictions.items():
    dmn_predictions[label] = add_transaction_costs(
        df, position_col="position", vol_col="ex_ante_vol", strat_col="strat_ret"
    )

# Apply to classical strategies
for name in ["long_only", "moskowitz", "macd"]:
    classical = classical.merge(
        stocks_oos[["date", "ticker", "60d_ewm_vol"]],
        on=["date", "ticker"], how="left",
    )
    classical = add_transaction_costs(
        classical,
        position_col=f"pos_{name}", vol_col="60d_ewm_vol",
        strat_col=f"strat_{name}",
    )

print("Transaction cost adjustment applied (turnover-based, paper Eq. C1)")

## Aggregate to portfolio level
 
Equal-weighted across stocks at each date — this is what the paper reports in Exhibits 3/4 (the strategy is defined per asset, then averaged).

In [ ]:
def to_portfolio_series(df: pd.DataFrame, strat_col: str) -> pd.Series:
    """Average strategy return across stocks per date."""
    return df.groupby("date")[strat_col].mean().sort_index()

portfolios = {}

for label, df in dmn_predictions.items():
    portfolios[f"{label}_gross"] = to_portfolio_series(df, "strat_ret")
    portfolios[f"{label}_net"]   = to_portfolio_series(df, "strat_ret_net")

for name in ["long_only", "moskowitz", "macd"]:
    portfolios[f"{name}_gross"] = to_portfolio_series(classical, f"strat_{name}")
    portfolios[f"{name}_net"]   = to_portfolio_series(classical, f"strat_{name}_net")

# SXXR daily returns (cap-weighted reference)
sxxr_prices = (benchmarks.loc[benchmarks["benchmark"] == "SXXR"]
                          .set_index("date")["price"].sort_index())
sxxr_returns = sxxr_prices.pct_change()
portfolios["sxxr"] = sxxr_returns.loc[(sxxr_returns.index >= oos_start) &
                                       (sxxr_returns.index <= oos_end)]

print(f"Portfolios computed: {len(portfolios)} series")

## Performance metrics
 
Replicates Exhibit 3 of the paper. All metrics are annualised, returns compound daily, drawdowns are continuous from peak.

In [ ]:
def compute_metrics(returns: pd.Series) -> dict:
    """Standard performance metrics for a daily strategy return series."""
    r = returns.dropna()
    if len(r) < 2:
        return {k: np.nan for k in ["Returns", "Vol", "Sharpe", "Downside Dev",
                                      "Sortino", "MDD", "Calmar", "% +ve",
                                      "Avg P / Avg L"]}
    
    ann_ret = r.mean() * 252
    ann_vol = r.std() * np.sqrt(252)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else np.nan
    
    downside = r[r < 0]
    dd_dev   = downside.std() * np.sqrt(252) if len(downside) > 0 else np.nan
    sortino  = ann_ret / dd_dev if dd_dev > 0 else np.nan
    
    cum  = (1 + r).cumprod()
    dd   = (cum - cum.cummax()) / cum.cummax()
    mdd  = dd.min()
    calmar = ann_ret / abs(mdd) if mdd != 0 else np.nan
    
    pct_pos = (r > 0).mean()
    avg_p   = r[r > 0].mean()
    avg_l   = abs(r[r < 0].mean())
    p_to_l  = avg_p / avg_l if avg_l > 0 else np.nan
    
    return {
        "Returns": ann_ret, "Vol": ann_vol, "Sharpe": sharpe,
        "Downside Dev": dd_dev, "Sortino": sortino,
        "MDD": mdd, "Calmar": calmar,
        "% +ve": pct_pos, "Avg P / Avg L": p_to_l,
    }

# Build the master metrics table (paper Exhibit 3)
metric_rows = []
for label, ret_series in portfolios.items():
    m = compute_metrics(ret_series)
    m["Strategy"] = label
    metric_rows.append(m)

metrics_df = pd.DataFrame(metric_rows).set_index("Strategy")
metrics_df = metrics_df[["Returns", "Vol", "Sharpe", "Downside Dev",
                          "Sortino", "MDD", "Calmar", "% +ve", "Avg P / Avg L"]]

# Pretty print
def format_metrics(df: pd.DataFrame) -> pd.DataFrame:
    fmt = df.copy()
    for col in ["Returns", "Vol", "Downside Dev", "MDD"]:
        fmt[col] = fmt[col].map(lambda x: f"{x:+.2%}" if pd.notna(x) else "—")
    for col in ["Sharpe", "Sortino", "Calmar", "Avg P / Avg L"]:
        fmt[col] = fmt[col].map(lambda x: f"{x:+.3f}" if pd.notna(x) else "—")
    fmt["% +ve"] = fmt["% +ve"].map(lambda x: f"{x:.1%}" if pd.notna(x) else "—")
    return fmt

print("\nFull-period out-of-sample metrics (paper Exhibit 3 format):")
print(format_metrics(metrics_df).to_string())

## Vol-rescaled metrics (paper Exhibit 4)
 
Each strategy is rescaled to a constant 15% annualised vol target. This makes risk-adjusted comparison clean (all strategies have the same vol).

In [ ]:
def rescale_to_target_vol(returns: pd.Series, target_vol=TARGET_VOL) -> pd.Series:
    """Scale daily returns so the realised annualised vol equals target_vol."""
    r = returns.dropna()
    if len(r) < 2: return r
    realised_vol = r.std() * np.sqrt(252)
    if realised_vol == 0: return r
    return r * (target_vol / realised_vol)

rescaled = {label: rescale_to_target_vol(s) for label, s in portfolios.items()}

# Same metrics, rescaled
rescaled_metric_rows = []
for label, ret_series in rescaled.items():
    m = compute_metrics(ret_series)
    m["Strategy"] = label
    rescaled_metric_rows.append(m)

rescaled_metrics_df = (pd.DataFrame(rescaled_metric_rows)
                          .set_index("Strategy")
                          [["Returns", "Vol", "Sharpe", "Downside Dev",
                            "Sortino", "MDD", "Calmar", "% +ve", "Avg P / Avg L"]])

print("\nVol-rescaled metrics (paper Exhibit 4 format, all at 15% vol):")
print(format_metrics(rescaled_metrics_df).to_string())

## Equity curves
 
Top: raw signal output. Bottom: vol-rescaled to 15% (so the curves are directly comparable on a risk-adjusted basis).

In [ ]:
PALETTE = {
    "dmn_baseline_net":     "#1f77b4",
    "dmn_cpd_net":          "#d62728",
    "dmn_cpd_longonly_net": "#9467bd",
    "long_only_net":        "#7f7f7f",
    "moskowitz_net":        "#2ca02c",
    "macd_net":             "#ff7f0e",
    "sxxr":                 "#000000",
}
LABELS = {
    "dmn_baseline_net":     "DMN (LSTM only)",
    "dmn_cpd_net":          "DMN w/ CPD",
    "dmn_cpd_longonly_net": "DMN w/ CPD long-only",
    "long_only_net":        "Long-only (EW)",
    "moskowitz_net":        "TSMOM (Moskowitz)",
    "macd_net":             "MACD",
    "sxxr":                 "SXXR (cap-weighted)",
}

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Raw signal (top)
for label, ret in portfolios.items():
    if label not in PALETTE: continue
    cum = (1 + ret.dropna()).cumprod()
    axes[0].plot(cum.index, cum.values, lw=1.2,
                  color=PALETTE[label], label=LABELS[label])
axes[0].set_yscale("log")
axes[0].set_title(f"Out-of-sample equity curves — raw signal "
                   f"({TRANSACTION_COST*1e4:.0f} bps cost)")
axes[0].set_ylabel("Cumulative return (log)"); axes[0].legend(loc="upper left", fontsize=9)
axes[0].axhline(1.0, color="gray", lw=0.5)

# Vol-rescaled (bottom)
for label, ret in rescaled.items():
    if label not in PALETTE: continue
    cum = (1 + ret.dropna()).cumprod()
    axes[1].plot(cum.index, cum.values, lw=1.2,
                  color=PALETTE[label], label=LABELS[label])
axes[1].set_yscale("log")
axes[1].set_title(f"Out-of-sample equity curves — rescaled to {TARGET_VOL:.0%} vol")
axes[1].set_ylabel("Cumulative return (log)"); axes[1].legend(loc="upper left", fontsize=9)
axes[1].axhline(1.0, color="gray", lw=0.5)

plt.tight_layout(); plt.show()

## Drawdown analysis
 
 Drawdown plot for the rescaled-to-15%-vol versions of each strategy.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
for label, ret in rescaled.items():
    if label not in PALETTE: continue
    cum = (1 + ret.dropna()).cumprod()
    dd = (cum - cum.cummax()) / cum.cummax()
    ax.plot(dd.index, dd.values, lw=1.0,
             color=PALETTE[label], label=LABELS[label], alpha=0.85)
ax.set_title(f"Drawdowns (rescaled to {TARGET_VOL:.0%} vol)")
ax.set_ylabel("Drawdown"); ax.legend(loc="lower left", fontsize=9)
ax.axhline(0, color="black", lw=0.5)
plt.tight_layout(); plt.show()

## Rolling Sharpe ratio
 
Trailing 252-day Sharpe to assess regime-dependence and the CPD module's value-add over time.

In [ ]:
def rolling_sharpe(returns: pd.Series, window: int = 252) -> pd.Series:
    r = returns.dropna()
    return (r.rolling(window).mean() / r.rolling(window).std()) * np.sqrt(252)

fig, ax = plt.subplots(figsize=(14, 5))
for label in ["dmn_baseline_net", "dmn_cpd_net", "dmn_cpd_longonly_net"]:
    if label not in portfolios: continue
    rs = rolling_sharpe(portfolios[label])
    ax.plot(rs.index, rs.values, lw=1.2,
             color=PALETTE[label], label=LABELS[label])

ax.axhline(0, color="black", lw=0.5)
ax.set_title("Trailing 252-day Sharpe ratio")
ax.set_ylabel("Rolling Sharpe (1y)"); ax.legend(loc="upper left", fontsize=9)
plt.tight_layout(); plt.show()

## Sharpe by year (and by sub-period)
 
Useful for documenting regime dependence — e.g. the paper notes the CPD module is especially valuable in 2015-2020. We replicate the cross-year breakdown here.

In [ ]:
def yearly_sharpe(returns: pd.Series) -> pd.Series:
    r = returns.dropna()
    return r.groupby(r.index.year).apply(
        lambda y: (y.mean() / y.std()) * np.sqrt(252) if y.std() > 0 else 0.0
    )

yearly_table = pd.DataFrame({
    LABELS[label]: yearly_sharpe(portfolios[label])
    for label in PALETTE if label in portfolios
})

fig, ax = plt.subplots(figsize=(14, 5))
yearly_table.plot(kind="bar", ax=ax, color=[PALETTE[l] for l in PALETTE if LABELS[l] in yearly_table.columns],
                   width=0.85, edgecolor="black", linewidth=0.4)
ax.set_title(f"Annual Sharpe ratio by strategy")
ax.set_ylabel("Sharpe (annualised)"); ax.set_xlabel("Year")
ax.axhline(0, color="black", lw=0.5)
ax.legend(fontsize=8, ncol=2, loc="lower left")
plt.tight_layout(); plt.show()

print("\nSharpe by year (one row per year):")
print(yearly_table.round(2).to_string())

## Transaction cost sensitivity
 
How does Sharpe degrade as we increase per-transaction cost from 0 bps to 50 bps? The paper notes (Appendix C) that DMNs deteriorate quickly above ~2 bps without a cost-aware loss; with 25 bps and a cost-aware model (our Option B), we expect more graceful degradation.

In [ ]:
COST_GRID_BPS = [0, 1, 2, 5, 10, 25, 50]   # in bps
COST_GRID = [c * 1e-4 for c in COST_GRID_BPS]

def sharpe_at_cost(df: pd.DataFrame, cost: float, position_col="position",
                    vol_col="ex_ante_vol", strat_col="strat_ret") -> float:
    """Re-derive Sharpe after subtracting turnover cost at a given level."""
    adj = add_transaction_costs(df, position_col=position_col,
                                  vol_col=vol_col, strat_col=strat_col,
                                  cost=cost)
    port = to_portfolio_series(adj, f"{strat_col}_net")
    return compute_metrics(port).get("Sharpe", np.nan)

cost_sens = {}
for label, df in dmn_predictions.items():
    cost_sens[label] = [sharpe_at_cost(df, c) for c in COST_GRID]

# For classical strategies the same calculation
for name in ["long_only", "moskowitz", "macd"]:
    sharpes = []
    for c in COST_GRID:
        adj = add_transaction_costs(classical,
                                       position_col=f"pos_{name}",
                                       vol_col="60d_ewm_vol",
                                       strat_col=f"strat_{name}",
                                       cost=c)
        port = to_portfolio_series(adj, f"strat_{name}_net")
        sharpes.append(compute_metrics(port).get("Sharpe", np.nan))
    cost_sens[name] = sharpes

cost_table = pd.DataFrame(cost_sens, index=COST_GRID_BPS)
cost_table.index.name = "Cost (bps)"

fig, ax = plt.subplots(figsize=(12, 5))
for label in cost_table.columns:
    color = PALETTE.get(label + "_net", PALETTE.get(label, None))
    ax.plot(cost_table.index, cost_table[label], marker="o", lw=1.5,
             label=LABELS.get(label + "_net", LABELS.get(label, label)),
             color=color)
ax.axhline(0, color="black", lw=0.5)
ax.set_xlabel("Average per-transaction cost (bps)")
ax.set_ylabel("Out-of-sample Sharpe ratio")
ax.set_title("Sharpe sensitivity to transaction cost  (paper Exhibit 8)")
ax.legend(fontsize=9, loc="upper right")
plt.tight_layout(); plt.show()

print("\nSharpe at each cost level:")
print(cost_table.round(3).to_string())

## Save final results
 
Tables and figures are exported to `data/processed/stoxx600/backtest/` for inclusion in the thesis writeup.

In [ ]:
out_dir = MODELS_DIR.parent / "backtest"
out_dir.mkdir(exist_ok=True, parents=True)

format_metrics(metrics_df).to_csv(out_dir / "metrics_raw.csv")
format_metrics(rescaled_metrics_df).to_csv(out_dir / "metrics_rescaled.csv")
yearly_table.round(2).to_csv(out_dir / "sharpe_by_year.csv")
cost_table.round(3).to_csv(out_dir / "cost_sensitivity.csv")

print(f"Saved tables to: {out_dir}")
print(f"  metrics_raw.csv         (paper Exhibit 3)")
print(f"  metrics_rescaled.csv    (paper Exhibit 4)")
print(f"  sharpe_by_year.csv      (regime breakdown)")
print(f"  cost_sensitivity.csv    (paper Exhibit 8)")

## Final commentary
 
**Key replication targets from the paper:**
 
- **Exhibit 3** (raw signal, paper p. 11): DMN with CPD should beat baseline DMN by ~30% in Sharpe (paper: 1.62 → 2.16). Compare to our `dmn_baseline_net` and `dmn_cpd_net` rows in `metrics_df`.

- **Exhibit 4** (rescaled to 15%): same Sharpe ranking, with returns now directly comparable in $-terms.

- **Exhibit 6** (equity curves): DMN with CPD should dominate visually, with the gap widening in 2015-2020 and 2022.

- **Exhibit 8** (cost sensitivity): paper's DMN deteriorates rapidly past 2 bps; with a cost-aware loss (`scripts/03_train_dmn.py --transaction-cost`) the curves should be much flatter.
 
**Practical deviations for our long-only equity fund:**
- The `dmn_cpd_longonly` variant constrains positions to $(0, 1)$, suitable for an unleveraged long-only mandate.
- 25 bps transaction costs (taxes included) reflects the realistic European cash equity environment for a fund of moderate size.
- For deployment, the Sharpe of `dmn_cpd_longonly_net` (with cost) is the honest metric to optimise; the gross / unconstrained variants are useful  as references but not as deployment targets.